# 2. 데이터 불러오기와 전처리

**단원 목표**

- CSV·엑셀 파일과 라이브러리 내장 데이터, 웹의 데이터를 불러오고 저장한다.
- 문자 인코딩을 이해하고 한글이 깨지는 문제를 해결한다.
- 결측치를 확인하고 상황에 맞게 삭제하거나 대체한다.
- 자료형 변환, 중복 제거, 열 이름과 문자열 정리로 데이터를 정제한다.
- 여러 데이터를 결합하고 wide 형식과 long 형식으로 바꾼다.
- 날짜·시간 데이터를 변환하고 기간별로 집계한다.

**실습 준비**

- 이 노트북은 같은 폴더의 `data` 폴더에 있는 실습 파일을 사용한다.
- **코랩**: 왼쪽 [파일] 창에 `data` 폴더를 통째로 끌어다 놓거나, 구글 드라이브에 올린 뒤 연결한다(코드 2-8 참고). 아래 셀로 pydataset을 설치한다.
- **자신의 PC(VS Code)**: 노트북과 같은 폴더에 `data` 폴더를 두고, 터미널에서 `pip install pydataset openpyxl`을 한 번 실행한 뒤 아래 셀은 건너뛴다.

In [ ]:
!pip install -q pydataset

## 2.1 데이터 불러오기와 저장

CSV·엑셀 파일과 내장 데이터, 웹 데이터를 불러오고, 한글 인코딩 문제를 해결하며, 데이터의 첫인상을 확인하고 저장한다.

### 2.1.1 CSV와 엑셀 파일 읽기

- **CSV**: 값을 쉼표로 구분한 텍스트 파일. 한 줄 = 한 행, 첫 줄 = 열 이름
- `pd.read_csv('경로', encoding=...)`: 기본 인코딩은 utf-8
- 국내 공공기관·엑셀 CSV는 **cp949**인 경우가 많다 → `UnicodeDecodeError`가 나면 `encoding='cp949'`
- 주요 인수: `encoding`, `sep`, `header`, `usecols`, `index_col`, `nrows`, `skiprows`, `na_values`
- 엑셀: `pd.read_excel('경로', sheet_name='시트')`, `sheet_name=None`이면 모든 시트를 딕셔너리로

In [ ]:
# 코드 2-1. CSV 파일 읽기
import pandas as pd

st = pd.read_csv('data/students_cp949.csv', encoding='cp949')
st

In [ ]:
# 코드 2-2. 인코딩을 지정하지 않았을 때의 오류
try:
    pd.read_csv('data/students_cp949.csv')       # 기본값 utf-8로 읽기
except UnicodeDecodeError as e:
    print('오류 발생:', e)

In [ ]:
# 코드 2-3. 필요한 열만 골라 읽기
st2 = pd.read_csv('data/students_cp949.csv', encoding='cp949',
                  usecols=['학번', '이름', '중간'], index_col='학번', nrows=5)
st2

In [ ]:
# 코드 2-4. 엑셀 파일 읽기
s1 = pd.read_excel('data/scores.xlsx', sheet_name='1학기')
print(s1.head(3))

sheets = pd.read_excel('data/scores.xlsx', sheet_name=None)   # 모든 시트
print(type(sheets), list(sheets.keys()))
print(sheets['2학기'].shape)

### 2.1.2 라이브러리 내장 데이터와 웹 데이터

- Seaborn: `sns.get_dataset_names()`, `sns.load_dataset('titanic')` (인터넷 필요)
- pydataset: `data()`로 목록, `data('airquality')`로 불러오기
- 웹 주소도 `read_csv(url)`로 바로 읽는다. 깃허브는 `raw.githubusercontent.com` 주소를 쓴다.
- 코랩에서 내 파일 쓰기: 파일 창에 올리기, `files.upload()`, `drive.mount()`

In [ ]:
# 코드 2-5. Seaborn 내장 데이터 불러오기
import seaborn as sns

print(sns.get_dataset_names()[:12])        # 데이터 목록 일부
titanic = sns.load_dataset('titanic')
print(titanic.shape)
titanic.head(3)

In [ ]:
# 코드 2-6. pydataset에서 데이터 찾아 불러오기
from pydataset import data

catalog = data()                                   # 전체 목록
print(catalog.shape)
print(catalog[catalog['title'].str.contains('Air')].head(4))

air = data('airquality')
air.head()

In [ ]:
# 코드 2-7. 웹 주소의 CSV 파일 읽기
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv'
penguins = pd.read_csv(url)
print(penguins.shape)
penguins.head(3)

In [ ]:
# 코드 2-8. 코랩에서 파일 올리기와 구글 드라이브 연결하기
# 방법 1: 내 PC의 파일을 코랩으로 올리기
from google.colab import files
uploaded = files.upload()          # 파일 선택 창이 열린다

# 방법 2: 구글 드라이브 연결하기
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/data/students_cp949.csv', encoding='cp949')

### 2.1.3 데이터 첫인상 확인과 저장

- 첫인상 확인: `head`, `tail`, `sample(n, random_state=…)`, `shape`, `info()`, `describe()`
- `info()`: 열별 결측치가 아닌 값의 개수(Non-Null Count)와 자료형(Dtype)
- 저장: `to_csv('경로', index=False, encoding='utf-8-sig')`, `to_excel('경로', index=False)`
  - `index=False`를 빼면 다시 읽을 때 `Unnamed: 0` 열이 생긴다.
  - 엑셀에서 열 CSV는 `utf-8-sig`로 저장해야 한글이 깨지지 않는다.

In [ ]:
# 코드 2-9. 마지막 행과 무작위 행 확인하기
cols = ['survived', 'pclass', 'sex', 'age', 'fare', 'embark_town']
print(titanic[cols].tail(3))
titanic[cols].sample(3, random_state=1)

In [ ]:
# 코드 2-10. info()로 구조 요약하기
titanic.info()

In [ ]:
# 코드 2-11. CSV와 엑셀 파일로 저장하기
import os
os.makedirs('output', exist_ok=True)

st['평균'] = st[['중간', '기말', '과제']].mean(axis=1).round(1)
st.to_csv('output/students_avg.csv', index=False, encoding='utf-8-sig')
st.to_excel('output/students_avg.xlsx', index=False, sheet_name='성적')

check = pd.read_csv('output/students_avg.csv')   # utf-8-sig 파일은 기본값으로 읽힌다
print(check.columns.tolist())
check.head(3)

> **[Data Focus] 문자 인코딩의 원리와 한글이 깨지는 이유**
>
> '가'는 UTF-8에서 3바이트(EA B0 80), CP949에서 2바이트(B0 A1)로 저장된다. 인코딩 오류가 나면 utf-8 → cp949 → utf-8-sig → latin-1 순서로 시도한다.

In [ ]:
# 코드 2-12. 인코딩에 따른 바이트 비교
print('가'.encode('utf-8'))                 # UTF-8: 3바이트
print('가'.encode('cp949'))                 # CP949: 2바이트
print('가나'.encode('utf-8').decode('cp949', errors='replace'))   # 잘못 해석

## 2.2 데이터 정제

결측치를 확인하고 처리하며, 자료형을 바꾸고 중복을 없애고, 열 이름과 문자열을 정리한다.

### 2.2.1 결측치 확인과 처리

- 확인: `df.isna().sum()`(개수), `df.isna().mean()`(비율)
- 삭제: `dropna()`, `dropna(subset=[…])`, `drop(columns=…)`
  - 결측치가 대부분인 열은 열을 지우고, 나머지는 필요한 열만 subset으로 지정한다.
- 대체: `fillna(값)` — 수치형은 중앙값·평균, 범주형은 최빈값 `mode()[0]`
- 시간 순서가 있는 데이터: `ffill()`, `bfill()`, `interpolate()`

In [ ]:
# 코드 2-13. 열별 결측치 개수와 비율 확인하기
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset('titanic')
na = pd.DataFrame({'결측치 수': titanic.isna().sum(),
                   '비율(%)': (titanic.isna().mean() * 100).round(1)})
na[na['결측치 수'] > 0].sort_values('결측치 수', ascending=False)

In [ ]:
# 코드 2-14. 결측치가 있는 행과 열 삭제하기
print('원본          :', titanic.shape)
print('모든 열 기준  :', titanic.dropna().shape)
print('age 열 기준   :', titanic.dropna(subset=['age']).shape)
print('deck 열 삭제  :', titanic.drop(columns='deck').dropna().shape)

In [ ]:
# 코드 2-15. 중앙값과 최빈값으로 대체하기
t2 = titanic.copy()
age_med = t2['age'].median()
emb_mode = t2['embarked'].mode()[0]
print('age 중앙값:', age_med, ' embarked 최빈값:', emb_mode)

t2['age'] = t2['age'].fillna(age_med)
t2['embarked'] = t2['embarked'].fillna(emb_mode)
print(t2[['age', 'embarked']].isna().sum())

In [ ]:
# 코드 2-16. 앞의 값과 보간으로 대체하기
from pydataset import data

air = data('airquality')
oz = air['Ozone'].iloc[:10]
pd.DataFrame({'원본': oz, 'ffill': oz.ffill(), 'interpolate': oz.interpolate()})

### 2.2.2 자료형 변환과 중복 제거

- `astype(int)`, `astype('category')`: 자료형 변환
- `pd.to_numeric(열, errors='coerce')`: 숫자로 바꿀 수 없는 값을 NaN으로
- 범주형(category)은 메모리를 줄이고 범주의 순서를 지정할 수 있다.
- 중복: `duplicated()`, `drop_duplicates(subset=…, keep='first'/'last')`

In [ ]:
# 코드 2-17. 숫자로 바꿀 수 없는 값이 섞인 열 변환하기
m = pd.DataFrame({'측정소': ['강남', '종로', '마포', '송파'],
                  '측정값': ['35', '42', '-', '28'],
                  '연도': [2025.0, 2025.0, 2025.0, 2025.0]})
print(m.dtypes)

m['측정값'] = pd.to_numeric(m['측정값'], errors='coerce')   # '-' → NaN
m['연도'] = m['연도'].astype(int)
print(m.dtypes)
m

In [ ]:
# 코드 2-18. 범주형으로 바꾸어 메모리 줄이기
diamonds = sns.load_dataset('diamonds')
color_str = diamonds['color'].astype(str)       # 문자열로 저장
color_cat = color_str.astype('category')        # 범주형으로 저장

print('문자열:', color_str.memory_usage(deep=True), '바이트')
print('범주형:', color_cat.memory_usage(deep=True), '바이트')
print(color_cat.cat.categories.tolist())

In [ ]:
# 코드 2-19. 중복 행 찾고 지우기
apply = pd.DataFrame({'회원': ['A01', 'A02', 'A02', 'A03', 'A03'],
                      '프로그램': ['요가', '수영', '수영', '헬스', '요가'],
                      '신청일': ['03-02', '03-02', '03-02', '03-03', '03-05']})
print(apply.duplicated())                          # 행 전체가 같은가?
print(apply.drop_duplicates().shape)
apply.drop_duplicates(subset='회원', keep='last')   # 회원별 마지막 신청만

### 2.2.3 열 이름과 문자열 다루기

- 열 이름: `df.columns.str.strip()`, `rename(columns={'옛 이름': '새 이름'})`
- 문자열 메서드는 `str`를 붙여 쓴다: `str.strip`, `str.replace`, `str.contains`, `str.split(expand=True)`, `str.extract(정규식)`
- '12,900원' → `str.replace(',', '').str.replace('원', '').astype(int)`

In [ ]:
# 코드 2-20. 열 이름 정리하기
shop = pd.DataFrame({' 상품명 ': ['사과 1kg', '배 3kg', ' 귤 5kg'],
                     '가격(원) ': ['12,900원', '25,000원', '18,500원'],
                     '판매처': ['서울시 강남구', '부산시 해운대구', '서울시 마포구']})
print(shop.columns.tolist())

shop.columns = shop.columns.str.strip()            # 앞뒤 공백 제거
shop = shop.rename(columns={'가격(원)': '가격'})
print(shop.columns.tolist())

In [ ]:
# 코드 2-21. 문자열 나누고 검색하기
shop[['시도', '구']] = shop['판매처'].str.split(' ', expand=True)
print(shop[['판매처', '시도', '구']])
print(shop[shop['판매처'].str.contains('서울')]['상품명'].str.strip().tolist())

In [ ]:
# 코드 2-22. 문자열을 숫자로 바꾸어 계산하기
shop['가격'] = shop['가격'].str.replace(',', '').str.replace('원', '').astype(int)
shop['무게'] = shop['상품명'].str.extract(r'(\d+)kg', expand=False).astype(int)
shop['kg당 가격'] = shop['가격'] // shop['무게']
shop[['상품명', '가격', '무게', 'kg당 가격']]

> **[Data Focus] 결측의 유형과 대체 방법의 선택**
>
> MCAR(완전 무작위), MAR(다른 변수와 관계), MNAR(빠진 값 자체와 관계). 타이타닉 deck의 결측은 객실 등급과 관계있어 MAR에 가깝다.

In [ ]:
# 코드 2-23. 객실 등급별 deck 결측 비율
print((titanic['deck'].isna().groupby(titanic['pclass']).mean() * 100).round(1))

## 2.3 데이터 결합·재구조화와 날짜 처리

여러 데이터를 결합하고 표의 모양을 바꾸며, 날짜·시간 데이터를 다룬 뒤 따릉이·기상 데이터를 정제한다.

### 2.3.1 데이터 결합

- `pd.concat([A, B], ignore_index=True)`: 위아래로 이어 붙이기, `axis=1`이면 좌우
- `pd.merge(A, B, on='키', how=…)`: 키 열을 기준으로 합치기
  - `inner`(양쪽 모두), `left`, `right`, `outer`(어느 한쪽이라도)
  - `indicator=True`: 각 행의 출처(_merge) 표시

In [ ]:
# 코드 2-24. 월별 파일 이어 붙이기
import pandas as pd

frames = []
for m in ['06', '07', '08']:
    df = pd.read_csv(f'data/bike_2018_{m}.csv')
    print(m, '월:', df.shape)
    frames.append(df)

summer = pd.concat(frames, ignore_index=True)
print('합계:', summer.shape)
summer[['Date', 'Hour', 'Rented Bike Count']].tail(3)

In [ ]:
# 코드 2-25. 결합 방식에 따른 merge 결과 비교하기
info = pd.DataFrame({'학번': [101, 102, 103, 104],
                     '이름': ['김민준', '이서연', '박지호', '최수아']})
score = pd.DataFrame({'학번': [102, 103, 104, 105],
                      '점수': [88, 75, 92, 67]})

for how in ['inner', 'left', 'outer']:
    print(how, pd.merge(info, score, on='학번', how=how).shape)
pd.merge(info, score, on='학번', how='outer', indicator=True)

### 2.3.2 데이터 재구조화

- **wide**: 한 행에 여러 값을 옆으로 (통계표), **long**: 값 하나가 한 행 (분석·그래프)
- wide → long: `melt(id_vars=…, var_name=…, value_name=…)`
- long → wide: `pivot(index=…, columns=…, values=…)` (조합이 중복되면 `pivot_table`)

In [ ]:
# 코드 2-26. wide 형식 표를 long 형식으로 바꾸기
import seaborn as sns

flights = sns.load_dataset('flights')
wide = flights.pivot(index='year', columns='month', values='passengers')
wide = wide.iloc[:3, :4].reset_index()          # 3개 연도, 4개 월만
wide.columns.name = None
print(wide)

long = wide.melt(id_vars='year', var_name='month', value_name='passengers')
long

In [ ]:
# 코드 2-27. long 형식을 wide 형식으로 되돌리기
back = long.pivot(index='year', columns='month', values='passengers')
back

### 2.3.3 날짜와 시간 데이터

- `pd.to_datetime(열, format='%d/%m/%Y')`: 형식을 반드시 지정한다.
- `dt` 접근자: `dt.year`, `dt.month`, `dt.dayofweek`(0=월), `dt.day_name()`
- `set_index('날짜').resample('MS').mean()`: 기간별 집계 ('D', 'W', 'MS', 'QS')

In [ ]:
# 코드 2-28. 형식을 지정해 날짜로 바꾸기
s = pd.Series(['01/06/2018', '02/06/2018', '13/06/2018'])
print(pd.to_datetime(s, format='%d/%m/%Y'))        # 일/월/연도로 해석
print(pd.to_datetime(s.iloc[:2], format='%m/%d/%Y')) # 월/일/연도로 해석

In [ ]:
# 코드 2-29. 날짜 만들고 요소 꺼내기
from pydataset import data

air = data('airquality')
air['date'] = pd.to_datetime(pd.DataFrame({'year': 1973, 'month': air['Month'],
                                           'day': air['Day']}))
air['dow'] = air['date'].dt.dayofweek           # 0=월요일
air['day_name'] = air['date'].dt.day_name()
air['week'] = air['date'].dt.isocalendar().week
air[['Month', 'Day', 'date', 'dow', 'day_name', 'week']].head()

In [ ]:
# 코드 2-30. 일별 데이터를 월별로 집계하기
monthly = air.set_index('date')[['Ozone', 'Temp']].resample('MS').mean().round(1)
monthly

> **[Data Focus] 병합 후 행 수가 늘어나는 이유**
>
> 키가 여러 번 나오면 짝이 되는 모든 조합이 만들어진다. `validate='one_to_one'` 등으로 기대하는 관계를 검사한다.

In [ ]:
# 코드 2-31. validate로 결합 관계 검사하기
score2 = pd.DataFrame({'학번': [102, 102, 103], '과목': ['통계', '파이썬', '통계'],
                       '점수': [88, 91, 75]})
print(pd.merge(info, score2, on='학번').shape)          # 1:N 결합 → 3행
try:
    pd.merge(info, score2, on='학번', validate='one_to_one')
except Exception as e:
    print('오류:', type(e).__name__, '-', e)

### 2.3.4 실제 데이터 적용: 서울 따릉이·기상 데이터 정제

- 데이터: UCI Seoul Bike Sharing Demand (2017.12.1~2018.11.30, 시간별 8,760행, CC BY 4.0)
- 정제 순서: 인코딩 지정(latin-1) → 열 이름 한글화 → 날짜 변환과 요일·주말 열 → 운영하지 않은 시간 제외 → 범주형 변환 → 저장(`bike_clean.csv`)
- `resample('D').agg({'대여량': 'sum', '기온': 'mean', '강수량': 'sum'})`로 일별 집계

In [ ]:
# 코드 2-32. 따릉이 데이터 읽기
raw = pd.read_csv('data/SeoulBikeData.csv', encoding='latin-1')
print(raw.shape)
print(raw.columns.tolist())

In [ ]:
# 코드 2-33. 열 이름을 한글로 바꾸기
bike = raw.copy()
bike.columns = ['날짜', '대여량', '시', '기온', '습도', '풍속', '가시거리',
                '이슬점', '일사량', '강수량', '적설량', '계절', '공휴일', '운영일']
bike.info()

In [ ]:
# 코드 2-34. 날짜 변환과 요일·주말 열 추가하기
bike['날짜'] = pd.to_datetime(bike['날짜'], format='%d/%m/%Y')
bike['요일'] = bike['날짜'].dt.dayofweek
bike['주말'] = bike['요일'].isin([5, 6])
print(bike['날짜'].min(), '~', bike['날짜'].max())
print(bike['날짜'].nunique(), '일 ×', bike['시'].nunique(), '시간')
bike[['날짜', '시', '대여량', '요일', '주말']].head(3)

In [ ]:
# 코드 2-35. 운영하지 않은 시간 확인하고 제외하기
print(bike['운영일'].value_counts())
print('운영 안 한 시간의 대여량 합계:', bike.loc[bike['운영일'] == 'No', '대여량'].sum())
print('운영 안 한 날짜 수:', bike.loc[bike['운영일'] == 'No', '날짜'].nunique())

bike = bike[bike['운영일'] == 'Yes'].drop(columns='운영일')
print('정제 후:', bike.shape, ' 중복 행:', bike.duplicated().sum())

In [ ]:
# 코드 2-36. 범주형 변환과 정제 데이터 저장
bike['계절'] = bike['계절'].astype('category')
bike['공휴일'] = (bike['공휴일'] == 'Holiday')            # True/False로 변환
bike.to_csv('data/bike_clean.csv', index=False, encoding='utf-8-sig')
print(bike.dtypes)

In [ ]:
# 코드 2-37. 일별 대여량과 날씨로 집계하기
daily = (bike.set_index('날짜')
             .resample('D')
             .agg({'대여량': 'sum', '기온': 'mean', '강수량': 'sum'})
             .round(1))
daily = daily[daily['대여량'] > 0]                      # 운영하지 않은 날 제외
print(daily.shape)
daily.sort_values('대여량', ascending=False).head(3)

## 단원 정리: 핵심 함수

| 구분 | 함수·메서드 | 기능 |
|---|---|---|
| 불러오기 | `read_csv`, `read_excel`, `sns.load_dataset`, `data` | 파일·내장 데이터 읽기 |
| 확인 | `head`, `tail`, `sample`, `info`, `describe` | 첫인상 확인 |
| 저장 | `to_csv(index=False, encoding='utf-8-sig')`, `to_excel` | 파일로 저장 |
| 결측치 | `isna`, `dropna`, `fillna`, `ffill`, `interpolate` | 확인·삭제·대체 |
| 자료형 | `astype`, `pd.to_numeric(errors='coerce')`, `'category'` | 변환 |
| 중복 | `duplicated`, `drop_duplicates` | 중복 확인·제거 |
| 문자열 | `str.strip`, `str.replace`, `str.contains`, `str.split`, `str.extract` | 문자열 정리 |
| 결합 | `pd.concat`, `pd.merge(how=, validate=)` | 이어 붙이기·합치기 |
| 재구조화 | `melt`, `pivot` | wide ↔ long |
| 날짜 | `pd.to_datetime(format=)`, `dt.*`, `resample` | 날짜 변환·기간 집계 |

연습문제는 `ch02_연습문제.ipynb`에서 풀어 본다.